<a href="https://colab.research.google.com/github/vifirsanova/ML-2026-pt-2/blob/main/rnn_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch RNN: полный туториал по классификации текста

## 1. Общая идея

В этом туториале мы разберём полный цикл работы с рекуррентной нейронной сетью на примере задачи классификации текста. Мы используем реальный открытый датасет **AG_NEWS** (классификация новостных тем, 4 класса, 120 тысяч обучающих примеров), потому что его можно автоматически скачать через `torchtext` — не нужно вручную подготавливать данные.

**Общий пайплайн:**
1. Автоматическая загрузка и исследование данных AG_NEWS
2. Предобработка текста (токенизация, построение словаря, паддинг последовательностей)
3. Построение RNN-классификатора (Embedding → RNN → Linear)
4. Обучение и валидация, построение кривых обучения
5. Оценка: матрица ошибок + classification report
6. **Три практических задания** (по 5–10 минут каждое), в которых вы сами меняете гиперпараметры и наблюдаете результат


In [ ]:
# Подготовка окружения
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchtext.datasets import AG_NEWS
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from tqdm import tqdm

# Фиксируем random seed
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Загрузка датасета AG_NEWS
# AG_NEWS возвращает кортежи (label, text), label — целое от 1 до 4
train_iter, test_iter = AG_NEWS(split=('train', 'test'))

# Преобразуем итераторы в списки для исследования
train_data = list(train_iter)
test_data = list(test_iter)

print(f"Обучающих примеров: {len(train_data)}")
print(f"Тестовых примеров: {len(test_data)}")
print(f"\nРаспределение меток (обучение): {Counter([label for label, _ in train_data])}")

In [ ]:
# Визуализация исследования данных
# Смотрим количество примеров по классам и распределение длин текстов
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Распределение меток
labels = [label for label, _ in train_data]
axes[0].bar(Counter(labels).keys(), Counter(labels).values(), color='steelblue')
axes[0].set_xlabel('Class Label')
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution (Train)')
axes[0].set_xticks([1, 2, 3, 4])

# Распределение длин текстов (первые 500 примеров)
lengths = [len(text.split()) for _, text in train_data[:500]]
axes[1].hist(lengths, bins=30, color='coral', edgecolor='white')
axes[1].set_xlabel('Text Length (words)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Text Length Distribution')

plt.tight_layout()
plt.show()

# Печатаем несколько примеров
print("\n=== Примеры ===")
for i in range(3):
    label, text = train_data[i]
    print(f"[Label {label}] {text[:150]}...")

## 3. Предобработка текста (область заданий)

In [ ]:
# Токенизация и построение словаря
# Здесь вы будете менять параметры — попробуйте изменить min_freq и посмотрите, как меняются размер словаря и точность

tokenizer = get_tokenizer('basic_english')

def yield_tokens(data_iter):
    for label, text in data_iter:
        yield tokenizer(text)

# ============================================
# Задание 1: параметры построения словаря
# Попробуйте min_freq = 2 или 10, посмотрите, как меняются размер словаря и качество модели
# Документация: https://pytorch.org/text/stable/vocab.html#build-vocab-from-iterator
# ============================================
MIN_FREQ = 5  # <-- меняйте здесь

vocab = build_vocab_from_iterator(
    yield_tokens(train_data),
    specials=['<unk>', '<pad>'],
    min_freq=MIN_FREQ
)
vocab.set_default_index(vocab['<unk>'])

PAD_IDX = vocab['<pad>']
print(f"Размер словаря: {len(vocab)}")

In [ ]:
# Числовое кодирование текста и обёртка Dataset
# Здесь вы будете менять параметры — попробуйте изменить MAX_LEN и посмотрите на результат

# ============================================
# Задание 2: длина последовательности (обрезка)
# Попробуйте MAX_LEN = 128, 256, 512, посмотрите на компромисс между скоростью обучения и точностью
# Подсказка: RNN дороже по вычислениям на длинных последовательностях, но длинные тексты могут содержать больше информации
# ============================================
MAX_LEN = 256  # <-- меняйте здесь

def text_pipeline(text):
    tokens = tokenizer(text)
    ids = [vocab[token] for token in tokens]
    if len(ids) > MAX_LEN:
        ids = ids[:MAX_LEN]
    return ids

def label_pipeline(label):
    return label - 1  # метки AG_NEWS начинаются с 1, приводим к 0-3

class AGNewsDataset(torch.utils.data.Dataset):
    def __init__(self, data, text_pipeline, label_pipeline, max_len):
        self.data = data
        self.text_pipeline = text_pipeline
        self.label_pipeline = label_pipeline
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        label, text = self.data[idx]
        ids = self.text_pipeline(text)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.label_pipeline(label), dtype=torch.long)

def collate_batch(batch):
    text_list, label_list = [], []
    for _text, _label in batch:
        text_list.append(_text)
        label_list.append(_label)
    # Паддинг до максимальной длины в батче с помощью PAD_IDX
    padded = nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=PAD_IDX)
    return padded, torch.stack(label_list)

train_dataset = AGNewsDataset(train_data, text_pipeline, label_pipeline, MAX_LEN)
test_dataset = AGNewsDataset(test_data, text_pipeline, label_pipeline, MAX_LEN)

In [ ]:
# Построение DataLoader
# Здесь вы будете менять параметры — попробуйте изменить BATCH_SIZE

# ============================================
# Задание 3: размер батча
# Попробуйте BATCH_SIZE = 32, 64, 128
# Документация: https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader
# ============================================
BATCH_SIZE = 64  # <-- меняйте здесь

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_batch, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_batch, num_workers=2)

print(f"Количество батчей в обучении: {len(train_loader)}")

## 4. Построение RNN-модели

In [ ]:
# RNN-классификатор
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=1, dropout=0.2, bidirectional=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        rnn_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc = nn.Linear(rnn_output_dim, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))  # (batch, seq_len, embed_dim)
        output, hidden = self.rnn(embedded)  # output: (batch, seq_len, hidden_dim)
        # Берём скрытое состояние последнего временного шага как представление предложения
        # hidden: (num_layers * num_directions, batch, hidden_dim)
        if self.rnn.bidirectional:
            last_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)  # (batch, hidden_dim*2)
        else:
            last_hidden = hidden[-1]  # (batch, hidden_dim)
        return self.fc(self.dropout(last_hidden))

In [ ]:
# Создание модели
# Здесь вы будете менять параметры — попробуйте изменить EMBED_DIM, HIDDEN_DIM, NUM_LAYERS

# ============================================
# Задание 4: гиперпараметры модели
# Попробуйте разные комбинации и посмотрите на точность на валидации
#   1. EMBED_DIM: 64, 128, 256
#   2. HIDDEN_DIM: 64, 128, 256
#   3. NUM_LAYERS: 1, 2
#   4. BIDIRECTIONAL: True / False
# Вопрос для рассуждения: как размерность скрытого состояния связана со способностью модели удерживать контекст?
# ============================================
EMBED_DIM = 128       # <-- меняйте здесь
HIDDEN_DIM = 128      # <-- меняйте здесь
NUM_LAYERS = 1        # <-- меняйте здесь
BIDIRECTIONAL = False # <-- меняйте здесь (True/False)
DROPOUT = 0.2

model = RNNClassifier(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=4,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    bidirectional=BIDIRECTIONAL
).to(device)

print(model)
print(f"Количество обучаемых параметров: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Обучение и валидация

In [ ]:
# Функция обучения
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct, total_samples = 0, 0, 0
    for texts, labels in tqdm(loader, desc="Training"):
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(texts)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)
    return total_loss / total_samples, total_correct / total_samples

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total_samples = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for texts, labels in tqdm(loader, desc="Evaluating"):
            texts, labels = texts.to(device), labels.to(device)
            logits = model(texts)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / total_samples, total_correct / total_samples, all_preds, all_labels

In [ ]:
# Цикл обучения
# Здесь вы будете менять параметры — попробуйте изменить скорость обучения и число эпох

# ============================================
# Задание 5: параметры оптимизатора
# Попробуйте LEARNING_RATE = 1e-3, 5e-4, 1e-4
# Подсказка: слишком большой learning rate может вызвать колебания loss; слишком маленький — медленную сходимость
# Документация: https://pytorch.org/docs/stable/optim.html#torch.optim.Adam
# ============================================
LEARNING_RATE = 1e-3  # <-- меняйте здесь
NUM_EPOCHS = 5        # <-- меняйте здесь

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, _, _ = evaluate(model, test_loader, criterion)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

## 6. Визуализация и анализ

In [ ]:
# Кривые обучения
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Матрица ошибок + classification report
val_loss, val_acc, all_preds, all_labels = evaluate(model, test_loader, criterion)

cm = confusion_matrix(all_labels, all_preds)
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

print("\n=== Classification Report ===")
print(classification_report(all_labels, all_preds, target_names=class_names))

## 7. Практические задания (по 5–10 минут каждое)

### Задание 1: Влияние размера словаря (5 минут)

**Цель:** понять, как `min_freq` влияет на размер словаря и качество модели.

**Действия:** вернитесь в Cell 4, измените `MIN_FREQ` на `2`, перезапустите эту ячейку и все последующие.

**Вопросы для рассуждения:**
- Насколько изменился размер словаря?
- Изменилась ли точность на валидации? Почему?
- Как вы думаете, что произойдёт с embedding-векторами редких слов, если `min_freq` сделать очень маленьким?

**Подсказка:** чем ниже `min_freq`, тем больше словарь, в него попадают редкие слова. Но редкие слова встречаются мало, embedding для них обучается плохо и может вносить шум. Документация: `torchtext.vocab.build_vocab_from_iterator`.

### Задание 2: Компромисс между длиной последовательности и RNN (8 минут)

**Цель:** проверить на практике компромисс между способностью RNN обрабатывать длинные последовательности и вычислительной стоимостью.

**Действия:** вернитесь в Cell 5, установите `MAX_LEN` равным `128` и `512`, обучите по одной эпохе и сравните время обучения и точность.

**Вопросы для рассуждения:**
- Как время обучения зависит от `MAX_LEN`?
- Всегда ли более длинная последовательность лучше?
- Если точность падает при увеличении `MAX_LEN`, с чем это может быть связано?

**Подсказка:** на длинных последовательностях у RNN есть проблема исчезающих градиентов. `512` может быть намного медленнее, чем `128`, а точность не обязательно вырастет. Реализация `torch.nn.utils.rnn.pad_sequence` определяет стоимость паддинга.

### Задание 3: Двунаправленный RNN и поиск гиперпараметров (10 минут)

**Цель:** попробовать процесс настройки параметров и найти конфигурацию лучше базовой.

**Действия:** в Cell 8 попробуйте следующие комбинации, обучая по одной эпохе (для быстрой итерации можно временно поставить `NUM_EPOCHS = 2`):

| Конфигурация | EMBED_DIM | HIDDEN_DIM | NUM_LAYERS | BIDIRECTIONAL |
|--------------|-----------|------------|------------|---------------|
| A            | 64        | 64         | 1          | False         |
| B            | 128       | 256        | 1          | True          |
| C            | 256       | 128        | 2          | False         |

**Вопросы для рассуждения:**
- Как количество параметров связано с точностью?
- Почему двунаправленный RNN обычно работает лучше? Что он видит такого, чего не видит обычный RNN?
- Что произойдёт с временем обучения при увеличении `NUM_LAYERS`?

**Подсказка:** выходная размерность двунаправленного RNN — `hidden_dim * 2`, но для финального скрытого состояния нужно конкатенировать `hidden[-2]` и `hidden[-1]` (последние состояния прямого и обратного проходов).

---

## 8. Вопросы для итогового рассуждения

После выполнения трёх заданий ответьте письменно на следующие вопросы:

1. Какая конфигурация гиперпараметров показала лучший результат на валидации? Почему, на ваш взгляд, именно она?
2. В чём принципиальное различие между обработкой текста с помощью CNN и с помощью RNN? Какие типы ошибок может допускать каждая из архитектур?
3. Как скрытое состояние `h_t` связано со способностью модели учитывать контекст? Что произойдёт, если его размерность сделать слишком маленькой?
4. Почему для длинных последовательностей RNN может терять информацию из начала текста? Как это наблюдается в ваших экспериментах?
5. Какие ограничения у RNN-подхода к классификации текста и в каких задачах они становятся критичными?
